# load the prediction json file

In [6]:
import os
import json
from sklearn.metrics import classification_report

In [7]:
pred_dir = 'predictions'

def load_predfile(pred_dir, mode, annot_type, vision_api):
    if vision_api:
        pred_filename = f'prediction_{mode}_{annot_type}_visionapi.json'
    else:
        pred_filename = f'prediction_{mode}_{annot_type}.json'
    pred_filepath = os.path.join(pred_dir, pred_filename)
    instances = []
    with open(pred_filepath, 'r') as predfile:
        lines = predfile.read().split("\n")[:-1]
        for line in lines:
            instance = json.loads(line)
            instances.append(instance)
    return instances

In [19]:
text_only_gold_instances = load_predfile(pred_dir, 'text_only', 'gold+silver', vision_api=False)
image_only_gold_instances = load_predfile(pred_dir, 'image_only', 'gold+silver', vision_api=False)
text_image_gold_instances = load_predfile(pred_dir, 'text_image', 'gold+silver', vision_api=False)

In [20]:
# print the classification report
def print_report(instances):
    gold_labels = [x['adjudicated_label'] for x in instances]
    pred_labels = [x['pred_label'] for x in instances]
    print(classification_report(gold_labels, pred_labels))
    print('-' * 60)

print("Classification report of text_only")
print_report(text_only_gold_instances)
print("Classification report of image_only")
print_report(image_only_gold_instances)
print("Classification report of text_image")
print_report(text_image_gold_instances)

Classification report of text_only
              precision    recall  f1-score   support

       Other       0.66      0.53      0.59       640
         Yes       0.62      0.74      0.67       668

    accuracy                           0.64      1308
   macro avg       0.64      0.63      0.63      1308
weighted avg       0.64      0.64      0.63      1308

------------------------------------------------------------
Classification report of image_only
              precision    recall  f1-score   support

       Other       0.62      0.51      0.56       640
         Yes       0.60      0.70      0.65       668

    accuracy                           0.61      1308
   macro avg       0.61      0.61      0.60      1308
weighted avg       0.61      0.61      0.60      1308

------------------------------------------------------------
Classification report of text_image
              precision    recall  f1-score   support

       Other       0.60      0.65      0.62       640
        

In [21]:
text_only_errors = [x for x in text_only_gold_instances if x['pred_label'] != x['adjudicated_label']]
image_only_errors = [x for x in image_only_gold_instances if x['pred_label'] != x['adjudicated_label']]
text_image_errors = [x for x in text_image_gold_instances if x['pred_label'] != x['adjudicated_label']]

In [22]:
text_only_errors[0]

{'Answer.Q1_annotator_0003': 'Other',
 'Answer.Q2_annotator_0003': '5',
 'adjudicated_label': 'Other',
 'HITId': '3R868ACW5Y0NJ69QNA8FL398QUOZGY',
 'image_filename': 'springbreak2019_anchor_1111303480461844480.jpg',
 'json_filename': 'springbreak2019_anchor_1111303480461844480.json',
 'anchor_location': 'Reno',
 'anchor_id': 'springbreak2019_1111303480461844480_1111303480461844480',
 'event': 'springbreak2019',
 'anchor_url': 'http://www.cse.unt.edu/~blanco/screenshots/anchor/springbreak2019_1111303480461844480_anchor_1111303480461844480.png',
 'tweet_text': 'Hoping everyone out there on spring break is enjoying some warm weather! 👕: #CuissedeGrenouille https://t.co/h4sJwFhQLm',
 'image_filepath': 'saved_anchors/image_file/springbreak2019_anchor_1111303480461844480.jpg',
 'tag_sequence': 'White, Font, Baby & toddler clothing, Active shirt',
 'annot_type': 'gold',
 'pred_label': 'Yes'}

In [23]:
def find_common_errors(errors1, errors2):
    res = []
    errors2_ids = [x['anchor_id'] for x in errors2]
    for error in errors1:
        if error['anchor_id'] in errors2_ids:
            res.append(error)
    return res

common_errors = find_common_errors(text_only_errors, image_only_errors)
common_error_ids = [x['anchor_id'] for x in common_errors]
errors_from_only_text = [x for x in text_only_errors if x['anchor_id'] not in common_error_ids]
errors_from_only_image = [x for x in image_only_errors if x['anchor_id'] not in common_error_ids]
print(f"# of errors made by both: {len(common_errors)}")
print(f"# of errors made by text_only: {len(errors_from_only_text)}")
print(f"# of errors made by image_only: {len(errors_from_only_image)}")

# of errors made by both: 234
# of errors made by text_only: 242
# of errors made by image_only: 279


In [24]:
len(image_only_errors)

513

### get the html code to display the errors

In [25]:
def write_html(errors, html_filename):
    i = 0
    with open(html_filename, 'w') as htmlfile:
        htmlfile.write("<!DOCTYPE html>\n<html>\n<head>\n<title>Error Example</title>\n</head>\n<body>\n")
        htmlfile.write("""<table style=\"margin-left: auto; margin-right: auto; border: 1px solid black; line-height: 1.0em; width: 1200\">\n""")
        image_css = """style=\"display: block; margin-left: auto; margin-right: auto; max-width: 450px; max-height: 650px;\""""
        
        while i < len(errors):
            
            htmlfile.write("<tr>\n")
            
            htmlfile.write("<td style=\"width:600; border-bottom: 1px solid black;\">\n")
            htmlfile.write(f"<img src=\"{errors[i]['anchor_url']}\" {image_css}>\n")
            htmlfile.write(f"<p style=\"text-align: center; margin: 10px\">Location: <strong>{errors[i]['anchor_location']}</strong>, Predicted label: <strong>{errors[i]['pred_label']}</strong>, Gold label: <strong>{errors[i]['adjudicated_label']}</strong></p>\n")
            htmlfile.write("</td>\n")
            
            i += 1
            
            if i < len(errors):
                htmlfile.write("<td style=\"width:600; border-bottom: 1px solid black;\">\n")
                htmlfile.write(f"<img src=\"{errors[i]['anchor_url']}\" {image_css}>\n")
                htmlfile.write(f"<p style=\"text-align: center; margin: 10px\">Location: <strong>{errors[i]['anchor_location']}</strong>, Predicted label: <strong>{errors[i]['pred_label']}</strong>, Gold label: <strong>{errors[i]['adjudicated_label']}</strong></p>\n")
                htmlfile.write("</td>\n")

                i += 1
        
        htmlfile.write("</table>\n")
        htmlfile.write("</body>\n</html>")  

only_text_errors = [x for x in text_only_errors if x not in common_errors]
only_image_errors = [x for x in image_only_errors if x not in common_errors]

common_errors_html_filename = 'common_errors.html'
only_text_errors_html_filename = 'only_text_errors.html'
only_image_errors_html_filename = 'only_image_errors.html'

write_html(common_errors, common_errors_html_filename)
write_html(only_text_errors, only_text_errors_html_filename)
write_html(only_image_errors, only_image_errors_html_filename)

## significance test

In [2]:
import os
import json
from statsmodels.stats.contingency_tables import mcnemar

In [5]:
pred_dir = 'predictions'
full_pred_filename = 'prediction_text_image_gold.json'
text_pred_filename = 'prediction_text_only_gold.json'
image_pred_filename = 'prediction_image_only_gold.json'
full_pred_filepath = os.path.join(pred_dir, full_pred_filename)
text_pred_filepath = os.path.join(pred_dir, text_pred_filename)
image_pred_filepath = os.path.join(pred_dir, image_pred_filename)

def load_predictions(pred_filepath):
    instances = {}
    with open(pred_filepath) as predfile:
        lines = predfile.read().split("\n")[:-1]
        for line in lines:
            instance = json.loads(line)
            instances[instance['anchor_id']] = instance
    return instances

full_preds = load_predictions(full_pred_filepath)
text_preds = load_predictions(text_pred_filepath)
image_preds = load_predictions(image_pred_filepath)

In [6]:
full_preds

{'electionday2019_1191843587580776448_1191843587580776448': {'Answer.Q1_annotator_0001': 'Yes',
  'Answer.Q2_annotator_0001': '5',
  'Answer.Q1_annotator_0002': 'Yes',
  'Answer.Q2_annotator_0002': '4',
  'adjudicated_label': 'Yes',
  'HITId': '3RTFSSG7UEUZB3IMNELOZRKY596WLB',
  'image_filename': 'electionday2019_anchor_1191843587580776448.jpg',
  'json_filename': 'electionday2019_anchor_1191843587580776448.json',
  'anchor_location': 'San Francisco',
  'anchor_id': 'electionday2019_1191843587580776448_1191843587580776448',
  'event': 'electionday2019',
  'anchor_url': 'http://www.cse.unt.edu/~blanco/screenshots/anchor/electionday2019_1191843587580776448_anchor_1191843587580776448.png',
  'tweet_text': 'It’s #electionday here is San Francisco. Make sure you get out and use your #vote 🗳 https://t.co/SByq5ckD6F',
  'image_filepath': 'saved_anchors/image_file/electionday2019_anchor_1191843587580776448.jpg',
  'tag_sequence': 'Blue, Brown, Denim, Textile, Collar, Pocket, White, Style, Elec

In [12]:
def sign_test(preds1, preds2):
    a, b, c, d = 0, 0, 0, 0
    for instance_id, instance in preds1.items():
        gold = instance['adjudicated_label']
        model_a = instance['pred_label']
        model_b = preds2[instance_id]['pred_label']
        if gold == model_a and gold == model_b:
            a += 1
        if gold == model_a and gold != model_b:
            b += 1
        if gold != model_a and gold == model_b:
            c += 1
        if gold != model_a and gold != model_b:
            d += 1
    table = [[a, b], [c, d]]
    result = mcnemar(table, exact=True)
    pvalue = result.pvalue
    return pvalue

# preds1, preds2 = text_preds, full_preds
preds1, preds2 = image_preds, image_preds
pvalue = sign_test(preds1, preds2)
print(f"p-value: {pvalue}")

p-value: 1.0
